# 01 — Target Definition & Merge Audit

Audit the `nyt_bestseller` label produced by `EDA/book_success_merge.ipynb`
and verify the merge is healthy before any modelling.

**Input:** `data/merged_books.csv`  
**Outputs:** class-balance report, match-quality spot-check, year-cohort table

## Goals
1. Confirm row count is stable (matches GoodBooks baseline after pre-1931 filter)
2. Inspect match method breakdown (exact ISBN vs fuzzy vs unmatched)
3. Check class balance overall and by publication decade
4. Understand NYT signal richness: `weeks_on_list`, `best_rank_achieved`, `debut_rank`
5. Choose train/test cutoff year based on cohort balance
6. Confirm leakage columns are correlated with the label (validates the target)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

DATA_PATH = Path('..') / 'data' / 'merged_books.csv'
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head(3)

## 1. Merge audit

In [ ]:
# gb_id must be unique — any duplication means the merge went wrong
assert df['gb_id'].is_unique, 'gb_id is not unique!'
print(f'Total rows: {len(df):,}')
print(f'Unique gb_id: {df["gb_id"].nunique():,}')
print()
print('Match method breakdown:')
print(df['match_method'].value_counts())

## 2. Class balance

In [ ]:
# Overall class balance
counts = df['nyt_bestseller'].value_counts().reindex([0, 1])
print('Class balance:')
print(counts)
print(f'\nPositive rate: {df["nyt_bestseller"].mean()*100:.1f}%')

# TODO: bar chart — non-bestseller vs bestseller count

In [ ]:
# Class balance by publication decade
df['pub_decade'] = (df['pub_year'] // 10 * 10).astype('Int64')
# TODO: plot bestseller rate by decade — flag any decade with < 5% or > 30%
#       as potentially problematic for modelling

## 3. NYT signal richness (bestsellers only)

In [ ]:
bs = df[df['nyt_bestseller'] == 1]
print(f'Bestsellers: {len(bs):,}')
bs[['weeks_on_list', 'best_rank_achieved', 'debut_rank', 'nyt_first_year']].describe()

In [ ]:
# TODO: histograms of weeks_on_list and best_rank_achieved
# NOTE: weeks_on_list and best_rank_achieved are valid for EDA but are
# NOT features — they are part of the outcome, not predictors

## 4. Choose train / test cutoff year

In [ ]:
# Print book count and positive rate for several candidate cutoffs.
# Good cutoff: test set has >= 200 positives and >= 5% positive rate.

for cutoff in [2005, 2008, 2010, 2012, 2014]:
    train = df[df['pub_year'] < cutoff]
    test  = df[df['pub_year'] >= cutoff]
    test_pos = test['nyt_bestseller'].sum()
    test_rate = test['nyt_bestseller'].mean() * 100
    print(f'Cutoff {cutoff}: train={len(train):,}  test={len(test):,}  '
          f'test_pos={test_pos}  test_rate={test_rate:.1f}%')

## 5. Leakage column sanity check

These columns **must not** enter the feature matrix, but we expect them to
correlate with `nyt_bestseller` — that confirms the label is picking up genuine
commercial success.

In [ ]:
LEAKAGE_COLS = [
    'average_rating', 'ratings_count', 'work_ratings_count',
    'work_text_reviews_count',
]
leakage_present = [c for c in LEAKAGE_COLS if c in df.columns]
# TODO: point-biserial correlation or boxplots of each leakage column vs nyt_bestseller
# Expected: bestsellers have higher ratings_count and slightly higher average_rating